# 19 — CT-BERT và Ensemble hai Transformer
**Câu hỏi:** thêm encoder chuyên miền COVID-19 có cải thiện Weighted-F1 không? Choquet có thêm lợi ích so với trung bình trên cùng hai thành viên không?
**Giao thức:** seed 42, fit 3.792, chọn epoch trên 702 mẫu nội bộ, calibration C, học kết hợp M. Giữ cả hai gap loss/F1 < 0,1. Một cấu hình, 3 epoch, tối đa 60 phút. Validation 2.139 chỉ xem sau khóa; không mở Test.

## 1. Kiểm tra đầu vào
Xác minh checksum split và checkpoint công khai; ghi cấu hình trước khi huấn luyện.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/ctbert_development.py').is_file())
sys.path.insert(0, str(root/'src'))
import ctbert_development as ct
ct.prepare()

## 2. Huấn luyện CT-BERT
Dùng preprocessing, độ dài, loss và quy tắc chọn epoch như đối chứng. Nếu không đạt cả hai gap, dừng trước đánh giá.

In [ ]:
ct.fit_neural(ct.KEY)

## 3. Khóa các phép kết hợp
Fit sigmoid chỉ trên C. Học Weighted Mean và Choquet chỉ trên M. So sánh thêm Equal Mean và pair-min cố định.

In [ ]:
ct.lock_parameters()

## 4. Đánh giá và lưu kết quả
Đối chiếu các phương pháp trên cùng ID và nhãn. Bảng hiển thị chênh lệch với DistilBERT; artifact lưu toàn bộ độ đo, ma trận nhầm lẫn và dự đoán.

In [ ]:
result = ct.evaluate_validation()
display(result[['model','weighted_f1','macro_f1','accuracy','mcc','log_loss','brier_real_binary','delta_weighted_f1_vs_distilbert']])